# ☁️ Topic 09 — Azure Fundamentals & Azure ML Workspace
> **Bootcamp:** Advanced ML / FAMA | Day 2 — Cloud & Scalable MLOps  
> **Target:** Master Azure cloud navigation, understand the Azure ML Workspace architecture, configure scalable compute, and manage assets with Python SDK v2.

---

### 🌟 Why Move Machine Learning to the Cloud?
On a local laptop:
* ⚠️ **Compute bottlenecks**: Training large models is constrained by local CPU/GPU and RAM limits.
* ⚠️ **Fragmented tracking**: Models and metrics live in ad-hoc local folders without governance.
* ⚠️ **Deployment friction**: Moving code from a laptop to production requires manual rewrites.

**The Azure ML Solution**: A centralized cloud workspace that provides managed auto-scaling compute, centralized data assets, automated experiment tracking via MLflow, a governed model registry, and managed zero-downtime inference endpoints.

## 🧭 1. Azure Core Concepts for ML Practitioners

Before diving into Azure ML, let's understand how Azure organizes cloud resources:

| Azure Concept | ML Analogy | Role in MLOps |
|---|---|---|
| **Subscription** | Master Billing Account | Boundary for financial budgets, cost tracking, and team quotas. |
| **Resource Group** | Project Namespace | Logical folder grouping all interrelated resources (Workspace, Storage, Key Vault). |
| **Region** | Physical Data Center | Geographic location (e.g. `East US`, `Southeast Asia`) chosen for compliance and latency. |
| **Storage Account** | Data Lake / Blob Store | Stores training datasets, experiment artifacts, and serialized model files (`.pkl`). |
| **Key Vault** | Secrets Locker | Safely stores database passwords, access tokens, and API keys. |
| **Container Registry (ACR)**| Docker Image Hub | Stores custom Docker images containing specialized ML dependencies. |
| **Application Insights** | Telemetry Watchtower | Captures real-time latency, request rates, error logs, and model performance.

## 🏢 2. The Azure Machine Learning Workspace

The **Azure ML Workspace** is the top-level foundational resource for all AI/ML activities. It acts as the central control plane connecting your data, compute clusters, experiments, and deployed models.

```
Azure ML Workspace
├── 🖥️ Compute
│   ├── Compute Instances (Interactive Jupyter / EDA)
│   ├── Compute Clusters (Auto-scaling multi-node training: 0 → N nodes)
│   └── Serverless Compute (Instant on-demand jobs)
├── 💾 Data
│   ├── Datastores (Secure links to Blob / ADLS Gen2)
│   └── Data Assets (Versioned dataset snapshots with lineage)
├── 🔬 Jobs
│   ├── Command Jobs (Single script training execution)
│   ├── Sweep Jobs (Automated hyperparameter search)
│   └── Pipeline Jobs (Multi-step end-to-end DAG workflows)
├── 📦 Models
│   └── Model Registry (Versioned models with staging & lineage)
├── 🐳 Environments
│   └── Curated & Custom Docker + Conda runtime environments
└── 🚀 Endpoints
    ├── Managed Online Endpoints (Real-time REST inference)
    └── Batch Endpoints (High-throughput bulk scoring)
```

In [ ]:
# TODO: Produce a readable plot or diagram plus a short interpretation of what it shows for API serving or container deployment and an Azure ML cloud job or asset workflow. Pay attention to the original cell's intended step: Set visual style.## Before you prompt your AI, think through:#   - What should this cell produce, and how will you know that it worked?#   - Which assumptions, risks, or design choices matter for this result?## What to prompt your AI:#   "I am working in a Jupyter notebook section called 🏢 2. The Azure Machine Learning Workspace. Implement API serving or container deployment and an Azure ML cloud job or asset workflow for this notebook section. Show me the reasoning first, then return runnable Python for this cell."## Context you MUST include:#   - You are working in the notebook section: 🏢 2. The Azure Machine Learning Workspace.#   - The notebook topic is \u2601\ufe0f Topic 09 \u2014 Azure Fundamentals & Azure ML Workspace\n",.#   - Use the variables, data, libraries, and file paths already established above.#   - Ask for an explanation of the reasoning and a runnable solution, not just a final answer.#   - Request code that fits this notebook's existing style and does not overwrite unrelated variables.#   - Mention the relevant variables or functions from this cell: ax, figsize, dpi, ws_box, boxstyle, pad, facecolor, edgecolor.## Once you have code, check it against:#   - The output type, columns, shape, or metric expected by the next section.#   - A small sanity check that would reveal an empty, impossible, or leaked result.

## 💻 3. Azure ML Compute Options & Cost Optimization

Understanding when to use each compute target is essential for managing cloud budgets:

| Compute Target | Scaling Behavior | Billed When | Recommended Workload |
|---|---|---|---|
| **Compute Instance** | Single dedicated VM (no auto-scale) | Whenever VM state is `Running` | Interactive EDA, debugging, Jupyter notebooks |
| **Compute Cluster** | Auto-scales **0 → N nodes** | **Only during active job execution** | Production training jobs, hyperparameter sweeps |
| **Serverless Compute** | Managed dynamic provisioning | Exact seconds of execution | Quick ad-hoc training scripts with zero cluster management |
| **Managed Online Endpoint** | Auto-scales based on traffic/CPU | 24/7 per provisioned replica | Real-time REST API serving (low latency) |
| **Batch Endpoint** | Automatically starts & stops compute | Only while scoring dataset | Nightly batch scoring jobs |

> 💰 **MLOps Cost Optimization Pro-Tip**:
> Set `min_instances = 0` and `idle_time_before_scale_down = 120` seconds on your training compute clusters. When training finishes, nodes automatically deprovision to **0**, reducing idle cloud costs to **$0.00**!

In [ ]:
# TODO: Produce a readable plot or diagram plus a short interpretation of what it shows for API serving or container deployment and an Azure ML cloud job or asset workflow. Pay attention to the original cell's intended step: Resource Group Box.## Before you prompt your AI, think through:#   - What should this cell produce, and how will you know that it worked?#   - Which assumptions, risks, or design choices matter for this result?## What to prompt your AI:#   "I am working in a Jupyter notebook section called 💻 3. Azure ML Compute Options & Cost Optimization. Implement API serving or container deployment and an Azure ML cloud job or asset workflow for this notebook section. Show me the reasoning first, then return runnable Python for this cell."## Context you MUST include:#   - You are working in the notebook section: 💻 3. Azure ML Compute Options & Cost Optimization.#   - The notebook topic is \u2601\ufe0f Topic 09 \u2014 Azure Fundamentals & Azure ML Workspace\n",.#   - Use the variables, data, libraries, and file paths already established above.#   - Ask for an explanation of the reasoning and a runnable solution, not just a final answer.#   - Request code that fits this notebook's existing style and does not overwrite unrelated variables.#   - Mention the relevant variables or functions from this cell: ax, figsize, dpi, rg_box, boxstyle, pad, facecolor, edgecolor.## Once you have code, check it against:#   - The output type, columns, shape, or metric expected by the next section.#   - A small sanity check that would reveal an empty, impossible, or leaked result.

## 🐍 4. Connecting via Azure ML Python SDK v2

Azure ML SDK v2 provides a streamlined, object-oriented interface for interacting with cloud resources.

### Authentication Strategy: `DefaultAzureCredential`
`DefaultAzureCredential` from `azure-identity` tries authentication mechanisms in the following priority:
1. **Environment Variables** (used in CI/CD pipelines via Service Principals: `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`, `AZURE_TENANT_ID`)
2. **Managed Identity** (used when running inside Azure VMs / Compute Instances)
3. **Azure CLI** (`az login` session on local developer machine)
4. **Interactive Browser** (prompts a web browser login modal)

```python
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Authenticate automatically
credential = DefaultAzureCredential()

# Initialize MLClient
ml_client = MLClient(
    credential=credential,
    subscription_id="<YOUR_SUBSCRIPTION_ID>",
    resource_group_name="<YOUR_RESOURCE_GROUP>",
    workspace_name="<YOUR_WORKSPACE_NAME>"
)
```

In [ ]:
# TODO: Produce runnable code, evaluation output, and a brief explanation of the result for an Azure ML cloud job or asset workflow and leakage-aware validation. Pay attention to the original cell's intended step: Check if Azure SDK is available and if live credentials are configured.## Before you prompt your AI, think through:#   - What should this cell produce, and how will you know that it worked?#   - Which assumptions, risks, or design choices matter for this result?## What to prompt your AI:#   "I am working in a Jupyter notebook section called Initialize MLClient. Implement an Azure ML cloud job or asset workflow and leakage-aware validation for this notebook section. Show me the reasoning first, then return runnable Python for this cell."## Context you MUST include:#   - You are working in the notebook section: Initialize MLClient.#   - The notebook topic is \u2601\ufe0f Topic 09 \u2014 Azure Fundamentals & Azure ML Workspace\n",.#   - Use the variables, data, libraries, and file paths already established above.#   - Ask for an explanation of the reasoning and a runnable solution, not just a final answer.#   - Request code that fits this notebook's existing style and does not overwrite unrelated variables.#   - Mention the relevant variables or functions from this cell: AZURE_SDK_AVAILABLE, subscription_id, resource_group, workspace_name, connected_live, credential, ml_client, ws.## Once you have code, check it against:#   - The output type, columns, shape, or metric expected by the next section.#   - A small sanity check that would reveal an empty, impossible, or leaked result.

In [ ]:
# TODO: Produce a readable plot or diagram plus a short interpretation of what it shows for an Azure ML cloud job or asset workflow and a visual analysis or architecture diagram. Pay attention to the original cell's intended step: Simulate 24-hour training workload cost comparison.## Before you prompt your AI, think through:#   - What should this cell produce, and how will you know that it worked?#   - Which assumptions, risks, or design choices matter for this result?## What to prompt your AI:#   "I am working in a Jupyter notebook section called Initialize MLClient. Implement an Azure ML cloud job or asset workflow and a visual analysis or architecture diagram for this notebook section. Show me the reasoning first, then return runnable Python for this cell."## Context you MUST include:#   - You are working in the notebook section: Initialize MLClient.#   - The notebook topic is \u2601\ufe0f Topic 09 \u2014 Azure Fundamentals & Azure ML Workspace\n",.#   - Use the variables, data, libraries, and file paths already established above.#   - Ask for an explanation of the reasoning and a runnable solution, not just a final answer.#   - Request code that fits this notebook's existing style and does not overwrite unrelated variables.#   - Mention the relevant variables or functions from this cell: hours, training_active, hourly_rate, cost_instance, cost_cluster, figsize, dpi, where.## Once you have code, check it against:#   - The output type, columns, shape, or metric expected by the next section.#   - A small sanity check that would reveal an empty, impossible, or leaked result.

In [ ]:
# TODO: Produce a readable plot or diagram plus a short interpretation of what it shows for an Azure ML cloud job or asset workflow and a visual analysis or architecture diagram. Pay attention to the original cell's intended step: Add score labels.## Before you prompt your AI, think through:#   - What should this cell produce, and how will you know that it worked?#   - Which assumptions, risks, or design choices matter for this result?## What to prompt your AI:#   "I am working in a Jupyter notebook section called Initialize MLClient. Implement an Azure ML cloud job or asset workflow and a visual analysis or architecture diagram for this notebook section. Show me the reasoning first, then return runnable Python for this cell."## Context you MUST include:#   - You are working in the notebook section: Initialize MLClient.#   - The notebook topic is \u2601\ufe0f Topic 09 \u2014 Azure Fundamentals & Azure ML Workspace\n",.#   - Use the variables, data, libraries, and file paths already established above.#   - Ask for an explanation of the reasoning and a runnable solution, not just a final answer.#   - Request code that fits this notebook's existing style and does not overwrite unrelated variables.#   - Mention the relevant variables or functions from this cell: features, on_prem_scores, azure_scores, y_pos, ax, figsize, dpi, bar_height.## Once you have code, check it against:#   - The output type, columns, shape, or metric expected by the next section.#   - A small sanity check that would reveal an empty, impossible, or leaked result.

## 📝 5. Key Takeaways & Knowledge Check

### 🔑 Essential Rules of Azure ML
1. **Workspace is the Single Pane of Glass**: Centralizes data, compute, experiment history, and model versions for complete traceability and audit compliance.
2. **Auto-Scale to Zero is King**: Always set `min_instances = 0` on training compute clusters so you only pay for compute when jobs are actively running.
3. **Four Companion Resources**: When provisioning a Workspace, Azure automatically creates an **Azure Storage Account**, **Key Vault**, **Container Registry**, and **Application Insights**.
4. **SDK v2 is Declarative**: Use clean, modern Python objects and YAML definitions for all MLOps automation and CI/CD pipelines.

---

### 🏋️ Hands-On Exercises for Learners

1. **Exercise 1 (Compute Cluster Config)**: Write an Azure ML SDK v2 snippet using `AmlCompute` to define a cluster named `gpu-cluster-large` with `Standard_NC12s_v3`, `min_instances = 0`, `max_instances = 4`, and `idle_time_before_scale_down = 180`.
2. **Exercise 2 (Datastore Registration)**: Identify the difference between a Datastore (connection credentials to storage) and a Data Asset (versioned reference to specific parquet/CSV files).

---

### ❓ Self-Check Quiz

1. **Q1: Which Azure service is used to store sensitive API keys and database connection strings used during ML training?**  
   *A: Azure Key Vault.*

2. **Q2: Why should you avoid using a Compute Instance to run a 10-hour distributed hyperparameter tuning job?**  
   *A: A Compute Instance is a single dedicated VM that does not scale out to parallel nodes and incurs costs 24/7 if not manually stopped.*

3. **Q3: What role does Azure Container Registry (ACR) play in Azure ML?**  
   *A: It stores the Docker images containing the exact Python runtime, libraries, and C++ packages required for training and deployment.*